In [34]:
# ============================================================
# File: 01_component1_vulnerability_intel_extraction.py
#
# Purpose:
#   Component 1 — Vulnerability Intelligence Extraction
#   Ingest raw CVE JSON files, normalize data, perform temporal
#   split (train/test), train calibrated ML model, and produce
#   probability-based severity predictions.
#
# Inputs:
#   data/raw/<YEAR>/CVE-XXXX-YYYY.json  (NVD dataset folders)
#
# Outputs:
#   outputs/intel_objects/ai_risk_intel_lightgbm_m2.jsonl
#   (structured AI_Risk_Intel objects)
#
# Notes:
#   - Dynamically detects PROJECT_ROOT.
#   - Uses LightGBM (optimized for M2 / CPU fallback).
#   - Includes SHAP-based feature interpretability.
# ============================================================

import os
import json
import logging
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
import shap
from scipy.sparse import hstack

# ------------------------------------------------------------
# 0. Project paths and logging
# ------------------------------------------------------------
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "intel_objects"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=str(OUTPUT_DIR / "component1_training.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("component1")

# ------------------------------------------------------------
# 1. Load and normalize NVD JSON records
# ------------------------------------------------------------
def load_all_nvd_records(raw_dir: Path, year_range=range(2020, 2026), limit=None) -> pd.DataFrame:
    records = []
    for year in year_range:
        ydir = raw_dir / str(year)
        if not ydir.exists():
            continue
        for file in ydir.glob("CVE-*.json"):
            try:
                with open(file, "r", encoding="utf-8") as f:
                    j = json.load(f)
                cve_id = j.get("id")
                desc = ""
                for d in j.get("cveTags", []):
                    if d.get("lang") == "en":
                        desc = d.get("value", "")
                cvss = j.get("baseScore") or j.get("metrics", {}).get("cvssMetricV31", [{}])[0].get("cvssData", {}).get("baseScore", None)
                sev = j.get("baseSeverity") or j.get("metrics", {}).get("cvssMetricV31", [{}])[0].get("cvssData", {}).get("baseSeverity", "UNKNOWN")
                if not cve_id or not desc:
                    continue
                records.append({"cve_id": cve_id, "description": desc, "cvss_score": cvss, "severity_band": sev})
            except Exception as e:
                logger.warning("Error parsing %s: %s", file.name, e)
        if limit and len(records) >= limit:
            break
    df = pd.DataFrame(records)
    df = df[df["severity_band"].isin(["LOW", "MEDIUM", "HIGH", "Low", "Medium", "High"])]
    return df

# ------------------------------------------------------------
# 2. Temporal split (train older, test newer)
# ------------------------------------------------------------
def temporal_split(df: pd.DataFrame, cutoff="2024-01-01"):
    df = df.copy()
    df["published"] = pd.to_datetime(df.get("published", datetime.now()))
    train = df[df["published"] < cutoff]
    test = df[df["published"] >= cutoff]
    return train, test

# ------------------------------------------------------------
# 3. Train LightGBM classifier
# ------------------------------------------------------------
def train_model(train_df, test_df):
    tfidf = TfidfVectorizer(max_features=1500, stop_words="english")
    X_train_text = tfidf.fit_transform(train_df["description"])
    X_test_text = tfidf.transform(test_df["description"])
    X_train = hstack([X_train_text, np.array(train_df["cvss_score"]).reshape(-1, 1)])
    X_test = hstack([X_test_text, np.array(test_df["cvss_score"]).reshape(-1, 1)])

    y_train = train_df["severity_band"].str.capitalize()
    y_test = test_df["severity_band"].str.capitalize()

    model = LGBMClassifier(objective="multiclass", num_class=3, n_estimators=200, random_state=42)
    model.fit(X_train.todense(), y_train, eval_set=[(X_test.todense(), y_test)], verbose=-1)

    print("=== Model Evaluation ===")
    print(classification_report(y_test, model.predict(X_test.todense())))
    return model, tfidf

# ------------------------------------------------------------
# 4. SHAP explainability + AI_Risk_Intel generation
# ------------------------------------------------------------
def make_ai_risk_intel(model, tfidf, df, outfile):
    explainer = shap.TreeExplainer(model, feature_perturbation="interventional", check_additivity=False)
    feature_names = list(tfidf.get_feature_names_out()) + ["cvss_score"]

    records = []
    for _, row in df.iterrows():
        x_vec = tfidf.transform([row["description"]])
        x_full = hstack([x_vec, np.array([[row["cvss_score"]]])])
        probs = model.predict_proba(x_full.todense())[0]
        sev_pred = model.predict(x_full.todense())[0]
        shap_vals = explainer.shap_values(x_full.todense())
        abs_vals = np.mean([np.abs(sv[0]) for sv in shap_vals], axis=0)
        top_idx = np.argsort(abs_vals)[-5:][::-1]
        top_tokens = [feature_names[i] for i in top_idx if i < len(feature_names)]

        rec = {
            "intel_id": row["cve_id"],
            "predicted_severity": sev_pred,
            "confidence_distribution": dict(zip(model.classes_, map(float, probs))),
            "key_features": top_tokens,
            "model_version": "1.0-LightGBM-M2",
            "timestamp": datetime.utcnow().isoformat()
        }
        records.append(rec)

    with open(outfile, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"Saved AI_Risk_Intel objects → {outfile}")

# ------------------------------------------------------------
# 5. Main execution
# ------------------------------------------------------------
if __name__ == "__main__":
    logger.info("Starting Component 1 pipeline.")
    df = load_all_nvd_records(DATA_RAW, year_range=range(1999, 2026))
    train_df, test_df = temporal_split(df)
    model, tfidf = train_model(train_df, test_df)
    OUT_FILE = OUTPUT_DIR / "ai_risk_intel_lightgbm_m2.jsonl"
    make_ai_risk_intel(model, tfidf, test_df, OUT_FILE)
    logger.info("Component 1 completed successfully.")

KeyError: 'severity_band'

In [ ]:
# ============================================================
# File: 01_component1_ai_risk_intel_sbert_gpu.py
#
# Description:
#   Component 1 — Vulnerability Intelligence Extraction (SBERT + LightGBM)
#   Automatically detects Apple M2 (MPS), CUDA, or CPU and adjusts accordingly.
#   Converts NVD JSON to calibrated AI_Risk_Intel objects for governance pipeline.
#
# Inputs:
#   data/raw/<year>/CVE-*.json
#
# Outputs:
#   outputs/intel_objects/ai_risk_intel_sbert_lgbm_gpu.jsonl
#   models/sbert_lgbm_gpu.pkl
#
# ============================================================

import os, json, warnings, joblib, shap, nltk, logging
import numpy as np, pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics.pairwise import cosine_similarity
from lightgbm import LGBMClassifier, log_evaluation
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Environment setup
# ------------------------------------------------------------
warnings.filterwarnings("ignore")
os.environ["LIGHTGBM_VERBOSITY"] = "-1"

# Dynamically resolve project root (portable)
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "intel_objects"
LOG_DIR      = PROJECT_ROOT / "outputs" / "logs"
MODEL_DIR    = PROJECT_ROOT / "models"
for d in [OUTPUT_DIR, LOG_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_DIR / "component1_sbert_lightgbm_gpu.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

nltk.download("wordnet", quiet=True)
nltk.download("stopwords", quiet=True)

# ------------------------------------------------------------
# 1. Parse NVD JSON
# ------------------------------------------------------------
def parse_nvd_json(fp: Path):
    """Safely parse one NVD CVE JSON file."""
    try:
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        if "cve" in data:
            cve = data["cve"]
            desc = next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")
            score = 0.0
            metrics = cve.get("metrics", {})
            for k in ["cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
                if k in metrics:
                    score = metrics[k][0]["cvssData"]["baseScore"]; break
            return {
                "cve_id": cve.get("id", ""),
                "description": desc,
                "cvss_score": score,
                "published": cve.get("published", data.get("published", "")),
            }
        return None
    except Exception as e:
        logging.warning(f"Parse error {fp.name}: {e}")
        return None

# ------------------------------------------------------------
# 2. Load NVD dataset
# ------------------------------------------------------------
def load_all_nvd_records(raw_dir: Path, year_range=None, limit=None):
    recs = []
    years = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
    if year_range:
        years = [p for p in years if int(p.name) in year_range]
    for y in years:
        for f in sorted(y.glob("CVE-*.json")):
            r = parse_nvd_json(f)
            if r: recs.append(r)
            if limit and len(recs) >= limit: break
        if limit and len(recs) >= limit: break
    return pd.DataFrame(recs)

# ------------------------------------------------------------
# 3. Temporal split
# ------------------------------------------------------------
def to_datetime_safe(s):
    try:
        return pd.to_datetime(s, utc=True, errors="coerce")
    except Exception:
        return pd.NaT

def temporal_split(df, test_ratio=0.25):
    df["published_ts"] = df["published"].apply(to_datetime_safe)
    df = df.sort_values("published_ts").reset_index(drop=True)
    cut = int((1 - test_ratio) * len(df))
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()

# ------------------------------------------------------------
# 4. SBERT embedding model (auto device)
# ------------------------------------------------------------
import torch
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"=== Loading SBERT model on {DEVICE.upper()} ===")
SBERT_MODEL = "all-MiniLM-L6-v2"
sbert = SentenceTransformer(SBERT_MODEL, device=DEVICE)
print("SBERT device:", sbert.device)

def encode_texts(model, texts):
    """Encode text using SBERT (batched)"""
    emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    return emb.astype(np.float32)

# ------------------------------------------------------------
# 5. Feature builder and model training
# ------------------------------------------------------------
def build_features(emb, cvss, scale=0.2):
    return np.hstack([emb, (scale * cvss.reshape(-1, 1)).astype(np.float32)])

def train_lgbm(Xtr, ytr, Xv, yv):
    classes = np.unique(ytr)
    w = compute_class_weight("balanced", classes=classes, y=ytr)
    cw = dict(zip(classes, w))

    # LightGBM on Apple M2 must run CPU mode (no CUDA)
    device_type = "gpu" if torch.cuda.is_available() else "cpu"
    model = LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        n_estimators=400,
        learning_rate=0.05,
        max_depth=-1,
        n_jobs=-1,
        device_type=device_type,
        class_weight=cw,
        random_state=42,
        verbosity=-1
    )
    model.fit(Xtr, ytr, eval_set=[(Xv, yv)], callbacks=[log_evaluation(0)])
    return model

# ------------------------------------------------------------
# 6. Concept bank for explainability
# ------------------------------------------------------------
SECURITY_CONCEPTS = [
    "buffer overflow","heap overflow","integer overflow","memory corruption",
    "race condition","cross site scripting","stored xss","reflected xss",
    "csrf","sql injection","command injection","authentication bypass",
    "authorization bypass","path traversal","directory traversal",
    "open redirect","file inclusion","remote code execution",
    "arbitrary code execution","privilege escalation","denial of service",
    "input validation","insufficient sanitization","improper access control"
]
concept_matrix = sbert.encode(SECURITY_CONCEPTS, normalize_embeddings=True)
def top_concepts(emb, k=5):
    sim = cosine_similarity(emb.reshape(1, -1), concept_matrix).ravel()
    return [SECURITY_CONCEPTS[i] for i in np.argsort(sim)[-k:][::-1]]

# ------------------------------------------------------------
# 7. Train pipeline
# ------------------------------------------------------------
df = load_all_nvd_records(RAW_DIR, year_range=range(1999, 2026))
df["description"] = df["description"].fillna("").astype(str)
df["cvss_score"] = df["cvss_score"].fillna(0.0)
df["severity_band"] = pd.cut(
    df["cvss_score"], [0, 4, 7, 10],
    labels=["Low", "Medium", "High"],
    include_lowest=True
).astype(str)

train_df, test_df = temporal_split(df)
print("Train/Test size:", len(train_df), len(test_df))

emb_train = encode_texts(sbert, train_df["description"].tolist())
emb_test = encode_texts(sbert, test_df["description"].tolist())

X_train = build_features(emb_train, train_df["cvss_score"].values)
X_test = build_features(emb_test, test_df["cvss_score"].values)
y_train, y_test = train_df["severity_band"], test_df["severity_band"]

clf = train_lgbm(X_train, y_train, X_test, y_test)
print("\n=== Model Evaluation ===")
print(classification_report(y_test, clf.predict(X_test)))

# ------------------------------------------------------------
# 8. Generate AI_Risk_Intel
# ------------------------------------------------------------
intel_records = []
for _, r in test_df.iterrows():
    emb = sbert.encode([r["description"]], normalize_embeddings=True)[0]
    X = build_features(emb.reshape(1, -1), np.array([r["cvss_score"]]))
    probs = clf.predict_proba(X)[0]
    top = top_concepts(emb, k=5)
    intel_records.append({
        "intel_id": r["cve_id"],
        "predicted_severity": clf.classes_[np.argmax(probs)],
        "prediction_distribution": dict(zip(clf.classes_, probs.round(3).tolist())),
        "key_influential_features": top,
        "model_metadata": {
            "model_version": "2.1-SBERT-LightGBM-M2",
            "timestamp": datetime.utcnow().isoformat(),
            "device": DEVICE
        }
    })

# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------
out_path = OUTPUT_DIR / "ai_risk_intel_sbert_lgbm_gpu.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for r in intel_records:
        f.write(json.dumps(r) + "\n")
joblib.dump(clf, MODEL_DIR / "sbert_lgbm_gpu.pkl")
print(f"\nSaved {len(intel_records)} AI_Risk_Intel records → {out_path}")

=== Loading SBERT model on MPS ===
